# Retrieval-Augmented Generation (RAG) with a Vector Database

In this notebook, you'll build a RAG pipeline from scratch. Instead of asking an LLM to answer from memory alone, you first **retrieve** the most relevant pieces of your own data, then **inject** that context into the prompt so the model can give an accurate answer.

You already know how embeddings work from the previous notebook. Here you'll put them to use in a complete pipeline:

1. **Load** a document and split it into chunks
2. **Embed** each chunk and store the vectors in a FAISS index
3. **Search** the index to find the most relevant chunks for a question
4. **Generate** an answer using only the retrieved context

In [ ]:
%pip install -q openai numpy faiss-cpu --upgrade

## 1. Setup

In [ ]:
import os
from getpass import getpass
import re
import faiss
import numpy as np
from openai import OpenAI

In [ ]:
# API Key Setup - uses environment variable or prompts for input
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

In [ ]:
# Client and Model Configuration
client = OpenAI()
MODEL = "gpt-5.2"
EMBEDDING_MODEL = "text-embedding-3-small"

## 2. Load and Chunk Your Document

RAG starts with your data. You'll load an article from a markdown file and split it into overlapping chunks using a **sliding window**. Overlapping chunks help ensure that important information spanning a sentence boundary isn't lost.

The `chunk_text` function below uses two parameters:
- **`window_size`**: how many sentences per chunk
- **`stride`**: how many sentences to advance between chunks (a stride smaller than the window creates overlap)

In [ ]:
def chunk_text(text, window_size=5, stride=2):
    """Split text into overlapping chunks using a sliding window over sentences."""
    sentences = re.split('(?<=[.!?]) +', text.strip()) # Split text into sentences based on punctuation
    chunks = []
    for i in range(0, len(sentences) - window_size + 1, stride): # Loop through the sentences and create chunks
        chunks.append(' '.join(sentences[i:i + window_size])) # Join the sentences into a chunk
    return chunks

# Load the article from a file instead of pasting it inline
with open("data/batman_history.md") as f:
    text = f.read()

chunks = chunk_text(text, window_size=4, stride=1)
print(f"Created {len(chunks)} chunks from the article\n")

# Preview the first chunk
print("Example chunk:")
print(chunks[0])

## 3. Build the Vector Index

Now you'll embed every chunk and store the vectors in a **FAISS** index. FAISS (Facebook AI Similarity Search) is a lightweight, in-memory vector database that's perfect for prototyping. For production workloads, you'd swap this out for something like Supabase pgvector, Pinecone, or Weaviate.

In [ ]:
def get_embedding(text: str, client: OpenAI) -> list[float]:
    response = client.embeddings.create(input=text, model=EMBEDDING_MODEL)
    return response.data[0].embedding


In [ ]:
# Embed all chunks
vectors = np.array([get_embedding(chunk, client) for chunk in chunks])
print(f"Embedded {vectors.shape[0]} chunks into {vectors.shape[1]}-dimensional vectors")

# Build a FAISS index for fast similarity search
index = faiss.IndexFlatL2(vectors.shape[1])
index.add(vectors)
print(f"FAISS index built with {index.ntotal} vectors")

In [ ]:
print(chunks)

## 4. Retrieve Relevant Chunks

This is the **R** in RAG. Given a question, you embed it into the same vector space, then find the closest chunks in your index. The results are the context that will ground your LLM's answer.

In [ ]:
# 1. Embed the query
# 2. Search the index and return the top k results

def vector_search(
    query_text: str,
    chunks: list[str],
    k: int = 3,
):
    query_vector = np.array([get_embedding(query_text, client)])
    distances, indicies = index.search(query_vector, k=k)

    # Loop over all of the chunks and get the relevant chunks specifically from the indicies:
    relevant_chunks = "\n".join([chunks[i] for i in indicies[0]])

    return distances, indicies, relevant_chunks


print(vector_search("batman", chunks, k=3))


## 5. Generate an Answer with Retrieved Context

This is the **G** in RAG. You take the retrieved chunks, inject them into a prompt as context, and ask the LLM to answer **only** from that context. The model doesn't need to have memorized the answer; it just needs to read the context you provide.

In [ ]:
def search_and_chat(question, chunks, k=3):
    # 1. Retrieve the top k results
    distances, indicies, relevant_chunks = vector_search(question, chunks, k)

    # 2. Generate the response
    response = client.responses.create(
        model=MODEL,
        instructions="""Answer the user's question based on the context provided. If the context does not contain the answer, say I don't know""",
        input=[
            {
                "role": "user",
                "content": "Context:\n"
                + relevant_chunks
                + "\n\nQuestion:\n"
                + question,
            }
        ],
    )

    # 3. Return the response
    print(f"Question: {question}")
    print(f"Answer: {response.output_text}")
    return response.output_text


# search_and_chat("Who is the actor who played Batman in the 2022 movie?", chunks, k=3)
search_and_chat("Who is James Phoenix?", chunks, k=3)

---

## Exercise 1: Ask Your Own Questions

Try asking 2-3 different questions about the Batman article using `search_and_chat()`. Experiment with questions the article **can** answer and questions it **can't**. Notice how the model responds differently in each case.

In [ ]:
# Try a question the article CAN answer:
search_and_chat("Your question here")

In [ ]:
# Try a question the article CANNOT answer:
search_and_chat("Your question here")

---

## Exercise 2: See the Effect of Context

What happens when you **remove** the retrieval step and ask the LLM directly? Run the cell below to compare a RAG answer vs. a no-context answer for the same question.

In [ ]:
question = "Which Batman movie killed the franchise?"

# With RAG — the model gets retrieved context
print("=== WITH RAG ===")
search_and_chat(question)

print("\n" + "=" * 60 + "\n")

# Without RAG — the model answers from its own training data only
print("=== WITHOUT RAG (no context) ===")
response = client.responses.create(
    model=MODEL,
    instructions="Answer the user's question. If you don't know, say 'I don't know'.",
    input=question
)
print(f"Question: {question}")
print(f"\nAnswer: {response.output_text}")

## Summary

In this notebook you built a complete RAG pipeline:

- **Chunking**: you split a document into overlapping pieces using a sliding window so no information is lost at sentence boundaries.
- **Indexing**: you embedded each chunk and stored the vectors in a FAISS index for fast similarity search.
- **Retrieval**: given a question, you found the most relevant chunks by searching the vector index.
- **Generation**: you injected the retrieved context into a prompt and had the LLM answer based on your data, not its training knowledge.

RAG lets you control what the model knows. By retrieving relevant context at query time, you can keep answers accurate, up-to-date, and grounded in your own documents without fine-tuning or retraining.